In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from datasets import load_dataset

C:\Users\ErdenebilegByambador\Projects\esg_analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = load_dataset("margiela00/task_dataset")["train"].to_pandas()
df = df.sort_values(["company_id", "year"]).reset_index(drop=True)
df.head(2)

,company_id,company_name,industry,region,year,revenue,profit_margin,market_cap,growth_rate,esg_overall_score,esg_environmental_score,esg_social_score,esg_governance_score,carbon_emissions,water_usage,energy_consumption
0,1,Company_1,Retail,Latin America,2015,459.2,6.0,337.5,NaN,57.0,60.7,33.5,76.8,35577.4,17788.7,71154.7
1,1,Company_1,Retail,Latin America,2016,473.8,4.6,366.6,3.2,56.7,58.9,32.8,78.5,37314.7,18657.4,74629.4


### Стратеги

Компаниудын зах зээлийн үнэлгээг таамаглах загвар сургана:  $M: X \rightarrow \mathbb{R}$ байх ба $X \in \mathbb{R}^{d}$ болон гаралтын $\mathbb{R}$ нь `market_cap`.

Үндсэн стратеги:
1. босго оноо тавих нэг форкаст модел сургана (M0).
2. бүх хувьсагчдийг оруулсан нэг комплекс форкаст модел сургана (M1).
3. Hypothesis тестээр `H0: Error(M0) = Error(M1)` Null таамаглалыг үгүйсгэж чадахгүй бол Occam's Razor-ын дүрмээр M0-ыг сонгоно.

Модел сонголттойгоор миний дүгнэлт:

- Нэг хугацааны цуваа (жнь: market cap) $\mathbb{R}^{1000 \times 11}$ матриц байг. Цаг хугацааны хувьд (longitudinal) богино ч цуваа олон байгаа нь (cross-sectional) ARIMA, ETS гэх мэт seasonality-д асар өндөр ач холбогдол өгдөг, удаан хугацааны өгөгдөл шаарддаг форкаст загвар сургах биш харин хөндлөн сургалт хийдэг (cross-learning) алгоритм ашиглах суурь аксиом болж байна. Иймд Gradient Boosting ашиглах LightGBM-ийг сонгож байна. 
- Форкастын алдааны үнэлгээ нь MAPE метрик гэж үзвэл $\mathbb{R}^{1000}$ 2 вектор дээр Wilcoxon тест гүйлгэж харах шаардлагатай. Жич: 2 вектор маань Гауссын тархалтаар байх албагүй тул T-Test ашиглахгүй гэж дүгнэлээ.
- M0-ийн хувьд LightGBM нь өнгөрсөн 3 жилийн лагийг аваад л ЗЗ-ийн үнэлгээг сайн таамаглаж чадвал илүү комплекс байлгах шаардлагагүй.
- M1-ийн хувьд санхүүгийн болон ESG, мөн статик датануудыг бүгдийг нь өгөөд сургаж үзэхээр төлөвлөв.
- __Хэрвээ цаг гарвал Transformer model ашиглана: Chronos-2 загвар богино цуваан дээр zero-shot хийх чадвартай хэлний загвар.__

In [ ]:
# M0: босго загвар: зөвхөн market_cap-ийн сүүлийн 3 жилийн лаг
# Баруун хазайлт өндөр (skew≈9) тул зорилтот хувьсагчийг лог руу шилжүүлнэ
# (market_cap > 0 тул энгийн np.log). lag1 нь сүүлийн түвшинг "зангуу" болгож
# өгснөөр модель лог-түвшнийг хэт экстраполяцлахгүй. Лагуудыг компаниар нь
# групплээд shift()-лэхэд хангалттай.
TARGET, N_LAGS = "market_cap", 3

panel = df[["company_id", "year", TARGET]].copy()
panel["y"] = np.log(panel[TARGET])
for k in range(1, N_LAGS + 1):
    panel[f"lag{k}"] = panel.groupby("company_id")["y"].shift(k)

panel = panel.dropna().reset_index(drop=True)   # лаг бүрдээгүй эхний 3 жил хасагдана
feat_cols = [f"lag{k}" for k in range(1, N_LAGS + 1)]

print("Panel:", panel.shape, "| зорилтот жилүүд:", panel.year.min(), "-", panel.year.max())
panel.head(3)

Panel: (8000, 7) | зорилтот жилүүд: 2018 - 2025


,company_id,year,market_cap,y,lag1,lag2,lag3
0,1,2018,283.0,5.645447,5.747480,5.904271,5.821566
1,1,2019,538.1,6.288044,5.645447,5.747480,5.904271
2,1,2020,384.1,5.950903,6.288044,5.645447,5.747480


In [66]:
# Хугацааны зааг: сүүлийн жил = тест, өмнөх он = сургалт. 1-алхамт урьдчилсан
# таамаг тул ирээдүйн мэдээлэл алдагдахгүй (no leakage). Тестийн алдааг компани
# тус бүрээр (R^1000) хадгалж M1-тэй Wilcoxon-оор харьцуулахад ашиглана.

TEST_YEAR = int(panel.year.max())
train = panel[panel.year < TEST_YEAR]
test = panel[panel.year == TEST_YEAR]

m0 = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05,
                         num_leaves=31, random_state=1, verbose=-1)
m0.fit(train[feat_cols], train["y"])

# Лог орон зайд таамаглаад MAPE-г ЖИНХЭНЭ нэгжид буцааж компани тус бүрээр авна.
pred = np.exp(m0.predict(test[feat_cols]))
true = np.exp(test["y"].to_numpy())
m0_ape = pd.Series(np.abs(pred - true) / true,
                   index=test["company_id"].to_numpy(), name="m0_ape")

print(f"M0 — тест жил {TEST_YEAR}, n={len(test)} компани")
print(f"MAPE: медиан={m0_ape.median():.1%}, дундаж={m0_ape.mean():.1%}")

M0 — тест жил 2025, n=1000 компани
MAPE: медиан=31.8%, дундаж=159.7%


Медиан < дундаж. Цөөн компанийн алдаа өндөр байгаа нь алдааны тархалтад right-skewness үүсгэж байна. Иймд төв хандлагыг **медианаар**, загвар хоорондын харьцуулалтыг rank-д суурилсан тестээр хийнэ. Дараагийн алхам: M1-д feature өргөтгөж, энэ сүүл болоод медиан сайжрах эсэхийг шалгана.

In [ ]:
# M1: комплекс загвар: бүх хувьсагчийн лаг + статик компани/салбар/бүс
# Зорилтот хувьсагч M0-той яг ижилхэн (log market_cap); цорын ганц ялгаа нь
# features-ийн тоо хэмжээ. Лог хувиргалтыг зөвхөн хазайлттай 5 баганад хийнэ;
# яагаад энэ 5-ыг сонгосныг мэдэх бол validation notebook руу буцаарай.
# ашиг/өсөлт/ESG оноонууд хязгаартай эсвэл тэмдэгтэй тул хэвээр.
NUM_COLS = ['revenue', 'profit_margin', 'market_cap', 'growth_rate',
            'esg_overall_score', 'esg_environmental_score', 'esg_social_score',
            'esg_governance_score', 'carbon_emissions', 'water_usage', 'energy_consumption']
LOG_COLS = ['revenue', 'market_cap', 'carbon_emissions', 'water_usage', 'energy_consumption']

d = df.sort_values(['company_id', 'year']).copy()
d['growth_rate'] = d['growth_rate'].fillna(0.0)   # 2015 суурь жил: өсөлт тодорхойгүй -> 0
for c in LOG_COLS:
    d[c] = np.log(d[c])

mp = d[['company_id', 'year']].copy()
mp['y'] = d['market_cap']                          # = log(market_cap), M0-той ижил
lag_cols = []
for c in NUM_COLS:
    for k in range(1, N_LAGS + 1):
        mp[f'{c}_lag{k}'] = d.groupby('company_id')[c].shift(k)
        lag_cols.append(f'{c}_lag{k}')
for c in ['industry', 'region', 'company_id']:
    mp[c] = d[c].astype('category')                # LightGBM статик категорийг өөрөө боловсруулна

mp = mp.dropna(subset=lag_cols).reset_index(drop=True)
feat_m1 = lag_cols + ['industry', 'region', 'company_id']
print("M1 panel:", mp.shape, "| features:", len(feat_m1),
      "| зорилтот жил:", mp.year.min(), "-", mp.year.max())

M1 panel: (8000, 38) | features: 36 | зорилтот жил: 2018 - 2025


In [ ]:
# M0-той яг ижил сүүлийн жилээ тестэд ашигласан ба гиперпараметр — ялгаа нь зөвхөн features.
train1 = mp[mp.year < TEST_YEAR]
test1 = mp[mp.year == TEST_YEAR]

m1 = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05,
                       num_leaves=31, random_state=1, verbose=-1)
m1.fit(train1[feat_m1], train1['y'])

pred1 = np.exp(m1.predict(test1[feat_m1]))
true1 = np.exp(test1['y'].to_numpy())
m1_ape = pd.Series(np.abs(pred1 - true1) / true1,
                   index=test1['company_id'].to_numpy(), name='m1_ape')

print(f"M1 — тест жил {TEST_YEAR}, n={len(test1)} компани")
print(f"MAPE: медиан={m1_ape.median():.1%}, дундаж={m1_ape.mean():.1%}")

M1 — тест жил 2025, n=1000 компани
MAPE: медиан=30.9%, дундаж=137.5%


Медиан бараг хэвээр (~1% APE), харин дундаж мэдэгдэхүйц буурсан → M1-ийн нэмэр нь төв нарийвчлал биш, **сүүлийн эрсдлийг (tail)** бууруулахад төвлөрнө. Дараах: энэ ялгаа санамсаргүй биш, найдвартай эсэхийг Wilcoxon-оор шалгана (ps: Student t-test-тэй адилхан гэхдээ error distribution Гауссын тархалттай байдаггүй болохоор Wilcoxon ашигласан). 

In [69]:
from scipy.stats import wilcoxon

# H0: Error(M0) = Error(M1). Алдааны вектор Гауссын биш (right skewed) тул
# rank-д суурилсан хосолсон Wilcoxon signed-rank ашиглана. alternative='greater':
# M0-ийн алдаа M1-ээс ИХ буюу M1 илүү нарийвчлалтай эсэхийг шалгана.
pair = pd.concat([m0_ape, m1_ape], axis=1).dropna()
stat, p = wilcoxon(pair['m0_ape'], pair['m1_ape'], alternative='greater')

print(f"M0 медиан APE = {pair['m0_ape'].median():.1%} | M1 медиан APE = {pair['m1_ape'].median():.1%}")
print(f"Wilcoxon signed-rank: statistic={stat:.0f}, p-value={p:.4g}")
if p < 0.05:
    print("=> H0-г үгүйсгэв: M1 статистикийн хувьд илүү нарийвчлалтай (комплекс нь зүйтэй).")
else:
    print("=> H0-г үгүйсгэж чадсангүй: Occam-аар M0-г сонгоно (нэмэлт feature үр өгөөжгүй).")

M0 медиан APE = 31.8% | M1 медиан APE = 30.9%
Wilcoxon signed-rank: statistic=290700, p-value=4.761e-06
=> H0-г үгүйсгэв: M1 статистикийн хувьд илүү нарийвчлалтай (комплекс нь зүйтэй).


## Дүгнэлт — статистик ач холбогдол vs практик ач холбогдол

H0 таамаглалыг үгүйсгэж M1 загварын нарийвчлал өндөр байгаа нь статистикийн хувьд хүчин төгөлдөр ч практик шийдвэрийн хувьд өөр гэж бодож байна. p-value нь тогтмол ялгаатай байдал нь үнэхээр хүчин төгөлдөр байна уу гэдгийг хэмждэг болохоос их хэмжээгээр ялгаатайг хэмждэггүй. Өөрөөр хэлбэл n=1000 дээр аль ч тогтмол өчүүхэн өндөр нарийвчлал "ач холбогдолтой" гарна.

- **Бодит хэмжээ өчүүхэн:** медиан 31.8% → 30.9% = ~**1 нэгж хувь**. 31% медиан алдаатай зах зээлийн үнэлгээний форкастад 1% APE бууралт ямар ч шийдвэрийг өөрчлөхгүй, M0=M1.
- **Occam:** M1 нь 33+ лаг, статик категори (бүр `company_id`-ийн цээжлэлт) нэмж байж дөнгөж +1% APE өгсөн — нэмэлт complexity, overfit болсон байдал ба сургах, ажиллуулах hassle гэх үү cost ч гэх үү тэр нэг юм нь хариуд нь өгөх үр өгөөжөөсөө хамаагүй даваад байна, ө.х. аягүй их features оруулаад2 өчүүхэн ROI.
- **Топ 100 зах зээлийн үнэлгээтэй компани дээр ч ахиц үзүүлээгүй:** Топ-100 зах зээлийн үнэлгээтэй компаний ID-г эрэмбэлж аваад 2 загварыг сургаж үзсэн. Зөрүү ~1% хэвээр (23% vs 22%) байсан болохоор цомхон байх үүднээс кодноос хасчихсан. Комплекс M1 загвар энэ топ 100-ийн error-ыг нэлээд өндөр хувиар бууруулсан байх гэж таамаглаж байсан нь үнэндээ үгүйсгэгдсэн.
- M1-ийн цорын ганц бодит нэмэр нь сүүлийн (хамгийн муу таамаглагдсан) компаниудын алдааг багасгах явдал (дундаж 159.7% → 137.5%). Хэрвээ маркет дээрх үнэлгээ нь муу эсвэл орлого муутай компаний error-ыг буулгахад төвлөрье гэвэл M1 дажгүй, гэхдээ тэрний use-case тийм олон гарч ирэхгүй байх.

**Шийдвэр: M0.** Статистик ялгаа практик утга агуулаагүй тул Occam Razor-ын зарчмаар энгийн загварыг сонгоно.

**Дараагийн алхам:** энэ нь нэг жилийн (2025) датаг тестэлсэн үр дүн. Rolling-origin backtest (2023/2024/2025) хийж дүгнэлт тогтвортой эсэхийг батлах; шаардвал MASE/RMSSE масштаб-чөлөөт метрик нэмэх.